### Retiro Fugas

In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-06-01'

filename='retiro_fugas.xlsx'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_excel(ruta_archivo)
df['NumDoc'] = df['Cliente'].str.extract(r'DNI-(\d+)')
df=df[df['Motivo']!='COBRANZA']
df['fecha'] = df['Fecha de creación'].dt.date
df['hora'] = df['Fecha de creación'].dt.time
df=df[[ 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Importe Solicitado', 'fecha','hora','NumDoc']]


server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	select NumDoc,TIPO_PRODUCTO,RETIRO
	from DANTALION.dbo.Base_Maestra_Diners_Vigente
"""
df_ssff = pd.read_sql(query, engine_kishin)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_ssff.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [77]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.48 seg
SP actualizar diners Zeus | realizado | duración: 5.54 seg
SP actualizar diners SA | realizado | duración: 1.5 seg
